In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATH
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "ALQ_C.xpt"
csv_file = DATA_DIR / "ALQ_C_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Keep untouched original for final fidelity comparison
source_df = df.copy()


print("=" * 70)
print("NHANES 2003-2004 ALQ_C XPT -> CSV")
print("=" * 70)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))

print("\nVariables:")
print(df.columns.tolist())


# ============================================================
# 3. EXPECTED RELEASED VARIABLES
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "ALQ101",
    "ALQ110",
    "ALQ120Q",
    "ALQ120U",
    "ALQ130",
    "ALQ140Q",
    "ALQ140U",
    "ALQ150"
]


if df.columns.tolist() == EXPECTED_COLUMNS:

    print(
        "\nPASS: Column names/order match ALQ_C."
    )

else:

    raise ValueError(
        "ALQ_C column structure differs from expected."
    )


# ============================================================
# 4. KNOWN PANDAS XPT ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)

zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print("\n--- BEFORE CORRECTION ---")

print(
    "True numeric zeros:",
    f"{int(zero_counts.sum()):,}"
)

print(
    "Tiny-value occurrences:",
    f"{int(tiny_counts.sum()):,}"
)


if tiny_counts.sum() > 0:

    print("\nTiny values by variable:")

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
    )


# ============================================================
# 5. ONLY THESE VARIABLES MAY HAVE ZERO RESTORED
# ============================================================

#
# CDC explicitly allows:
#
# ALQ120Q = 0 -> Never
# ALQ140Q = 0 -> None
#

ZERO_ALLOWED_COLUMNS = [
    "ALQ120Q",
    "ALQ140Q"
]


# ============================================================
# 6. SAFETY CHECK
# ============================================================

actual_tiny_columns = set(
    tiny_counts[
        tiny_counts > 0
    ].index
)

allowed_tiny_columns = set(
    ZERO_ALLOWED_COLUMNS
)


unexpected_tiny = (
    actual_tiny_columns
    -
    allowed_tiny_columns
)


if unexpected_tiny:

    raise ValueError(
        "STOP: Tiny values found in unexpected variables: "
        + str(sorted(unexpected_tiny))
    )


print(
    "\nPASS: Tiny values occur only in "
    "CDC zero-permitted ALQ variables."
)


# ============================================================
# 7. RESTORE TRUE ZEROS
# ============================================================

print("\n--- ZERO RESTORATION ---")


for col in ZERO_ALLOWED_COLUMNS:

    count = int(
        df[col]
        .eq(TINY_VALUE)
        .sum()
    )

    print(
        f"{col}: {count:,} tiny values -> 0"
    )

    df.loc[
        df[col].eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 8. VERIFY SOURCE-DERIVED ZERO COUNTS
# ============================================================

#
# From THIS attached XPT:
#
# ALQ120Q tiny values = 980
# ALQ140Q tiny values = 1705
#
# After restoration these should become true zeros.
#

print("\n--- ZERO COUNT AFTER RESTORATION ---")

print(
    "ALQ120Q zeros:",
    int(df["ALQ120Q"].eq(0).sum())
)

print(
    "ALQ140Q zeros:",
    int(df["ALQ140Q"].eq(0).sum())
)


if int(df["ALQ120Q"].eq(0).sum()) == 980:

    print(
        "PASS: ALQ120Q restored 980 zeros."
    )

else:

    raise ValueError(
        "ALQ120Q zero restoration count differs."
    )


if int(df["ALQ140Q"].eq(0).sum()) == 1705:

    print(
        "PASS: ALQ140Q restored 1,705 zeros."
    )

else:

    raise ValueError(
        "ALQ140Q zero restoration count differs."
    )


# ============================================================
# 9. VERIFY NO GENUINE DECIMALS EXIST
# ============================================================

#
# Do NOT blindly convert values to integers.
#
# First verify every nonmissing value is whole-number.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


for col in numeric_cols:

    values = df[col].dropna()

    if values.empty:
        continue

    fractional = ~np.isclose(
        values,
        np.round(values)
    )


    if fractional.any():

        print(
            f"Fractional values found in {col}:"
        )

        print(
            values[
                fractional
            ].head(20)
        )

        raise ValueError(
            f"{col} contains genuine decimal values. "
            "Column will NOT be changed."
        )


print(
    "PASS: All ALQ values are whole-number "
    "codes/quantities."
)


# ============================================================
# 10. CLEAN SEQN
# ============================================================

df["SEQN"] = (
    df["SEQN"]
    .round()
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


# ============================================================
# 11. FORMAT ALQ VALUES WITHOUT .0
# ============================================================

#
# Since we have already verified there are
# NO genuine fractional values:
#
# 1.0   -> 1
# 365.0 -> 365
# 999.0 -> 999
# 0.0   -> 0
#

ALQ_COLUMNS = [
    col
    for col in df.columns
    if col != "SEQN"
]


for col in ALQ_COLUMNS:

    df[col] = (
        df[col]
        .round()
        .astype("Int64")
    )


# ============================================================
# 12. CDC STRUCTURE CHECK
# ============================================================

print("\n--- CDC STRUCTURE CHECK ---")


if len(df) == 4742:

    print(
        "PASS: Row count = 4,742"
    )

else:

    print(
        "FAIL: Unexpected row count:",
        len(df)
    )


if len(df.columns) == 9:

    print(
        "PASS: Column count = 9"
    )

else:

    print(
        "FAIL: Column count =",
        len(df.columns)
    )


# ============================================================
# 13. CDC FREQUENCY CHECK - ALQ101
# ============================================================

expected = {
    1: 2956,
    2: 1356,
    9: 1
}

actual = (
    df["ALQ101"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- ALQ101 ---")

if (
    actual == expected
    and
    int(df["ALQ101"].isna().sum()) == 429
):

    print(
        "PASS: ALQ101 matches CDC."
    )

else:

    print(
        "FAIL: ALQ101 differs."
    )

    print("Actual:", actual)


# ============================================================
# 14. ALQ110
# ============================================================

expected = {
    1: 724,
    2: 631,
    9: 2
}

actual = (
    df["ALQ110"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- ALQ110 ---")

if (
    actual == expected
    and
    int(df["ALQ110"].isna().sum()) == 3385
):

    print(
        "PASS: ALQ110 matches CDC."
    )

else:

    print(
        "FAIL: ALQ110 differs."
    )


# ============================================================
# 15. ALQ120Q
# ============================================================

#
# CDC:
#
# 0-365 = 3675 records
# 777   = 0
# 999   = 4
# missing = 1063
#

s = df["ALQ120Q"]

regular = int(
    s.between(
        0,
        365
    ).sum()
)

dont_know = int(
    s.eq(999).sum()
)

refused = int(
    s.eq(777).sum()
)

missing = int(
    s.isna().sum()
)


print("\n--- ALQ120Q ---")

print(
    "0-365:",
    regular
)

print(
    "777:",
    refused
)

print(
    "999:",
    dont_know
)

print(
    "Missing:",
    missing
)

print(
    "True zeros:",
    int(s.eq(0).sum())
)


if (
    regular == 3675
    and
    refused == 0
    and
    dont_know == 4
    and
    missing == 1063
):

    print(
        "PASS: ALQ120Q matches CDC."
    )

else:

    print(
        "FAIL: ALQ120Q differs from CDC."
    )


# ============================================================
# 16. ALQ120U
# ============================================================

expected = {
    1: 1114,
    2: 661,
    3: 920
}

actual = (
    df["ALQ120U"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- ALQ120U ---")

if (
    actual == expected
    and
    int(df["ALQ120U"].isna().sum()) == 2047
):

    print(
        "PASS: ALQ120U matches CDC."
    )

else:

    print(
        "FAIL: ALQ120U differs."
    )


# ============================================================
# 17. ALQ130
# ============================================================

#
# CDC:
#
# 1-50 = 2696
# 77   = 0
# 99   = 3
# missing = 2043
#

s = df["ALQ130"]

regular = int(
    s.between(
        1,
        50
    ).sum()
)

refused = int(
    s.eq(77).sum()
)

dont_know = int(
    s.eq(99).sum()
)

missing = int(
    s.isna().sum()
)


print("\n--- ALQ130 ---")

if (
    regular == 2696
    and
    refused == 0
    and
    dont_know == 3
    and
    missing == 2043
):

    print(
        "PASS: ALQ130 matches CDC."
    )

else:

    print(
        "FAIL: ALQ130 differs."
    )


# ============================================================
# 18. ALQ140Q
# ============================================================

#
# CDC:
#
# 0-365 = 2696
# 777   = 0
# 999   = 3
# missing = 2043
#

s = df["ALQ140Q"]

regular = int(
    s.between(
        0,
        365
    ).sum()
)

refused = int(
    s.eq(777).sum()
)

dont_know = int(
    s.eq(999).sum()
)

missing = int(
    s.isna().sum()
)


print("\n--- ALQ140Q ---")

print(
    "True zeros:",
    int(s.eq(0).sum())
)


if (
    regular == 2696
    and
    refused == 0
    and
    dont_know == 3
    and
    missing == 2043
):

    print(
        "PASS: ALQ140Q matches CDC."
    )

else:

    print(
        "FAIL: ALQ140Q differs."
    )


# ============================================================
# 19. ALQ140U
# ============================================================

expected = {
    1: 189,
    2: 150,
    3: 652
}

actual = (
    df["ALQ140U"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- ALQ140U ---")

if (
    actual == expected
    and
    int(df["ALQ140U"].isna().sum()) == 3751
):

    print(
        "PASS: ALQ140U matches CDC."
    )

else:

    print(
        "FAIL: ALQ140U differs."
    )


# ============================================================
# 20. ALQ150
# ============================================================

expected = {
    1: 600,
    2: 3079
}

actual = (
    df["ALQ150"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- ALQ150 ---")

if (
    actual == expected
    and
    int(df["ALQ150"].isna().sum()) == 1063
):

    print(
        "PASS: ALQ150 matches CDC."
    )

else:

    print(
        "FAIL: ALQ150 differs."
    )


# ============================================================
# 21. FINAL TINY VALUE CHECK
# ============================================================

remaining_tiny = int(
    df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print("\n--- FINAL ZERO FIDELITY ---")

print(
    "Remaining tiny values:",
    remaining_tiny
)


if remaining_tiny == 0:

    print(
        "PASS: No tiny zero artifact remains."
    )

else:

    raise ValueError(
        "FAIL: Tiny values remain."
    )


# ============================================================
# 22. BUILD EXPECTED SOURCE FOR FIDELITY CHECK
# ============================================================

expected_df = source_df.copy()


for col in ZERO_ALLOWED_COLUMNS:

    expected_df.loc[
        expected_df[col].eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 23. VERIFY VALUES BEFORE EXPORT
# ============================================================

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )

    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if same:

        print(
            f"PASS: {col:<10} preserved."
        )

    else:

        raise ValueError(
            f"FAIL: {col} changed."
        )


# ============================================================
# 24. EXPORT CSV
# ============================================================

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 25. RE-READ CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False
)


# ============================================================
# 26. XPT -> CSV VALUE FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )

    csv_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        csv_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if same:

        print(
            f"PASS: {col:<10} matches XPT."
        )

    else:

        failures.append(
            col
        )

        print(
            f"FAIL: {col} differs."
        )


if not failures:

    print(
        "\nPASS: Every ALQ_C value is preserved "
        "after validated zero restoration."
    )


# ============================================================
# 27. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    source_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )

    csv_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            source_missing
            !=
            csv_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches == 0:

    print(
        "PASS: Missing values preserved exactly."
    )


# ============================================================
# 28. RAW CSV FORMAT CHECK
# ============================================================

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count == 0:

    print(
        "PASS: No unnecessary .0 formatting."
    )

else:

    print(
        "CHECK: .0 formatting remains."
    )


print("\n" + "=" * 70)
print("ALQ_C CONVERSION COMPLETE")
print("=" * 70)

NHANES 2003-2004 ALQ_C XPT -> CSV
Rows: 4,742
Columns: 9

Variables:
['SEQN', 'ALQ101', 'ALQ110', 'ALQ120Q', 'ALQ120U', 'ALQ130', 'ALQ140Q', 'ALQ140U', 'ALQ150']

PASS: Column names/order match ALQ_C.

--- BEFORE CORRECTION ---
True numeric zeros: 0
Tiny-value occurrences: 2,685

Tiny values by variable:
ALQ140Q    1705
ALQ120Q     980
dtype: int64

PASS: Tiny values occur only in CDC zero-permitted ALQ variables.

--- ZERO RESTORATION ---
ALQ120Q: 980 tiny values -> 0
ALQ140Q: 1,705 tiny values -> 0

--- ZERO COUNT AFTER RESTORATION ---
ALQ120Q zeros: 980
ALQ140Q zeros: 1705
PASS: ALQ120Q restored 980 zeros.
PASS: ALQ140Q restored 1,705 zeros.

--- FRACTIONAL VALUE CHECK ---
PASS: All ALQ values are whole-number codes/quantities.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 21009
Maximum: 31125

--- CDC STRUCTURE CHECK ---
PASS: Row count = 4,742
PASS: Column count = 9

--- ALQ101 ---
PASS: ALQ101 matches CDC.

--- ALQ110 ---
PASS: ALQ110 matches CDC.

--- ALQ120Q ---
0-365: 3675
7